In [0]:
df = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True, inferSchema=True)
display(df.limit(10))

raceId,driverId,lap,position,time,milliseconds
841,20,1,1,1:38.109,98109
841,20,2,1,1:33.006,93006
841,20,3,1,1:32.713,92713
841,20,4,1,1:32.803,92803
841,20,5,1,1:32.342,92342
841,20,6,1,1:32.605,92605
841,20,7,1,1:32.502,92502
841,20,8,1,1:32.537,92537
841,20,9,1,1:33.240,93240
841,20,10,1,1:32.572,92572


In [0]:
display(dbutils.fs.ls("/Volumes/gr5069/raw/f1_data/"))

path,name,size,modificationTime
dbfs:/Volumes/gr5069/raw/f1_data/circuits.csv,circuits.csv,10104,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/constructor_results.csv,constructor_results.csv,219365,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/constructor_standings.csv,constructor_standings.csv,317206,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/constructors.csv,constructors.csv,17478,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/driver_standings.csv,driver_standings.csv,883771,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/drivers.csv,drivers.csv,94367,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/lap_times.csv,lap_times.csv,17622395,1774389976000
dbfs:/Volumes/gr5069/raw/f1_data/pit_stops.csv,pit_stops.csv,443719,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/qualifying.csv,qualifying.csv,465231,1774389974000
dbfs:/Volumes/gr5069/raw/f1_data/races.csv,races.csv,164344,1774389974000


In [0]:
results_df = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
display(results_df.limit(10))

resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1
6,18,6,3,8,13,6,6,6,3.0,57,\N,\N,50,14,1:29.639,212.974,11
7,18,7,5,14,17,7,7,7,2.0,55,\N,\N,54,8,1:29.534,213.224,5
8,18,8,6,1,15,8,8,8,1.0,53,\N,\N,20,4,1:27.903,217.180,5
9,18,9,2,4,2,\N,R,9,0.0,47,\N,\N,15,9,1:28.753,215.100,4
10,18,10,7,12,18,\N,R,10,0.0,43,\N,\N,23,13,1:29.558,213.166,3


In [0]:
print(results_df.columns)
print(results_df.count())

['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId']
26759


In [0]:
results_df.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: string (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: string (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: string (nullable = true)
 |-- fastestLap: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: string (nullable = true)
 |-- statusId: integer (nullable = true)



In [0]:
display(
    results_df.select(
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "positionOrder",
        "points",
        "laps",
        "fastestLap",
        "rank",
        "fastestLapSpeed",
        "statusId"
    ).limit(20)
)

raceId,driverId,constructorId,grid,positionOrder,points,laps,fastestLap,rank,fastestLapSpeed,statusId
18,1,1,1,1,10.0,58,39,2,218.300,1
18,2,2,5,2,8.0,58,41,3,217.586,1
18,3,3,7,3,6.0,58,41,5,216.719,1
18,4,4,11,4,5.0,58,58,7,215.464,1
18,5,1,3,5,4.0,58,43,1,218.385,1
18,6,3,13,6,3.0,57,50,14,212.974,11
18,7,5,17,7,2.0,55,54,8,213.224,5
18,8,6,15,8,1.0,53,20,4,217.180,5
18,9,2,2,9,0.0,47,15,9,215.100,4
18,10,7,18,10,0.0,43,23,13,213.166,3


In [0]:
from pyspark.sql.functions import col, when

model_df = (
    results_df
    .select(
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "positionOrder",
        "points",
        "laps",
        "fastestLap",
        "rank",
        "fastestLapSpeed",
        "statusId"
    )
    .withColumn("target", when(col("positionOrder") <= 10, 1).otherwise(0))
)

display(model_df.limit(20))

raceId,driverId,constructorId,grid,positionOrder,points,laps,fastestLap,rank,fastestLapSpeed,statusId,target
18,1,1,1,1,10.0,58,39,2,218.300,1,1
18,2,2,5,2,8.0,58,41,3,217.586,1,1
18,3,3,7,3,6.0,58,41,5,216.719,1,1
18,4,4,11,4,5.0,58,58,7,215.464,1,1
18,5,1,3,5,4.0,58,43,1,218.385,1,1
18,6,3,13,6,3.0,57,50,14,212.974,11,1
18,7,5,17,7,2.0,55,54,8,213.224,5,1
18,8,6,15,8,1.0,53,20,4,217.180,5,1
18,9,2,2,9,0.0,47,15,9,215.100,4,1
18,10,7,18,10,0.0,43,23,13,213.166,3,1


In [0]:
model_df.printSchema()

root
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- grid: integer (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- fastestLap: string (nullable = true)
 |-- rank: string (nullable = true)
 |-- fastestLapSpeed: string (nullable = true)
 |-- statusId: integer (nullable = true)
 |-- target: integer (nullable = false)



In [0]:
display(model_df.summary())

summary,raceId,driverId,constructorId,grid,positionOrder,points,laps,fastestLap,rank,fastestLapSpeed,statusId,target
count,26759,26759,26759,26759,26759,26759,26759,26759,26759,26759,26759,26759
mean,551.6872827833627,278.6735304009866,50.18053738928958,11.134795769647596,12.794050599798199,1.9876321985126502,46.30176762958257,42.7329132331556,10.334312573443007,204.1163303441589,17.224971037781682,0.42307261108412125
stddev,313.2650358778931,282.7030394590417,61.55149811898847,7.202859614279887,7.665950638690448,4.351208910645763,29.496556967717673,16.603459536904044,6.1409568000890475,21.377264763058594,26.026103940899368,0.49405596714736333
min,1,1,1,0,1,0.0,0,1,0,100.615,1,0
25%,300,57,6,5,6,0.0,23,33.0,5.0,193.31,1,0
50%,531,172,25,11,12,0.0,53,46.0,10.0,204.885,10,0
75%,811,399,63,17,18,2.0,66,54.0,15.0,217.309,14,1
max,1144,862,215,34,39,50.0,200,\N,\N,\N,141,1


In [0]:
clean_df = (
    model_df
    .dropna(subset=[
        "grid",
        "laps",
        "fastestLap",
        "rank",
        "fastestLapSpeed",
        "statusId"
    ])
)

print("Original rows:", model_df.count())
print("Clean rows:", clean_df.count())

display(clean_df.limit(20))

Original rows: 26759
Clean rows: 26759


raceId,driverId,constructorId,grid,positionOrder,points,laps,fastestLap,rank,fastestLapSpeed,statusId,target
18,1,1,1,1,10.0,58,39,2,218.300,1,1
18,2,2,5,2,8.0,58,41,3,217.586,1,1
18,3,3,7,3,6.0,58,41,5,216.719,1,1
18,4,4,11,4,5.0,58,58,7,215.464,1,1
18,5,1,3,5,4.0,58,43,1,218.385,1,1
18,6,3,13,6,3.0,57,50,14,212.974,11,1
18,7,5,17,7,2.0,55,54,8,213.224,5,1
18,8,6,15,8,1.0,53,20,4,217.180,5,1
18,9,2,2,9,0.0,47,15,9,215.100,4,1
18,10,7,18,10,0.0,43,23,13,213.166,3,1


In [0]:
final_df = clean_df.select(
    "grid",
    "laps",
    "fastestLap",
    "rank",
    "fastestLapSpeed",
    "statusId",
    "target"
)

display(final_df.limit(20))

grid,laps,fastestLap,rank,fastestLapSpeed,statusId,target
1,58,39,2,218.300,1,1
5,58,41,3,217.586,1,1
7,58,41,5,216.719,1,1
11,58,58,7,215.464,1,1
3,58,43,1,218.385,1,1
13,57,50,14,212.974,11,1
17,55,54,8,213.224,5,1
15,53,20,4,217.180,5,1
2,47,15,9,215.100,4,1
18,43,23,13,213.166,3,1


In [0]:
display(final_df.groupBy("target").count())

target,count
1,11321
0,15438


In [0]:
from pyspark.sql.functions import expr

final_df_fixed = final_df.select(
    "grid",
    "laps",
    expr("try_cast(nullif(fastestLap, '\\\\N') as double)").alias("fastestLap"),
    expr("try_cast(nullif(rank, '\\\\N') as double)").alias("rank"),
    expr("try_cast(nullif(fastestLapSpeed, '\\\\N') as double)").alias("fastestLapSpeed"),
    "statusId",
    "target"
)

final_df_fixed = final_df_fixed.dropna(subset=["fastestLap", "rank", "fastestLapSpeed"])

final_df_fixed.printSchema()
print("Rows after cleaning:", final_df_fixed.count())
display(final_df_fixed.limit(10))

root
 |-- grid: integer (nullable = true)
 |-- laps: integer (nullable = true)
 |-- fastestLap: double (nullable = true)
 |-- rank: double (nullable = true)
 |-- fastestLapSpeed: double (nullable = true)
 |-- statusId: integer (nullable = true)
 |-- target: integer (nullable = false)

Rows after cleaning: 8252


grid,laps,fastestLap,rank,fastestLapSpeed,statusId,target
1,58,39.0,2.0,218.3,1,1
5,58,41.0,3.0,217.586,1,1
7,58,41.0,5.0,216.719,1,1
11,58,58.0,7.0,215.464,1,1
3,58,43.0,1.0,218.385,1,1
13,57,50.0,14.0,212.974,11,1
17,55,54.0,8.0,213.224,5,1
15,53,20.0,4.0,217.18,5,1
2,47,15.0,9.0,215.1,4,1
18,43,23.0,13.0,213.166,3,1


In [0]:
from pyspark.sql.functions import expr

final_df_fixed = clean_df.select(
    "grid",
    "laps",
    expr("try_cast(nullif(fastestLap, '\\\\N') as double)").alias("fastestLap"),
    expr("try_cast(nullif(rank, '\\\\N') as double)").alias("rank"),
    expr("try_cast(nullif(fastestLapSpeed, '\\\\N') as double)").alias("fastestLapSpeed"),
    "statusId",
    "target"
)

final_df_fixed = final_df_fixed.dropna(subset=["fastestLap", "rank", "fastestLapSpeed"])

final_df_fixed.printSchema()
print("Rows after cleaning:", final_df_fixed.count())
display(final_df_fixed.limit(10))

root
 |-- grid: integer (nullable = true)
 |-- laps: integer (nullable = true)
 |-- fastestLap: double (nullable = true)
 |-- rank: double (nullable = true)
 |-- fastestLapSpeed: double (nullable = true)
 |-- statusId: integer (nullable = true)
 |-- target: integer (nullable = false)

Rows after cleaning: 8252


grid,laps,fastestLap,rank,fastestLapSpeed,statusId,target
1,58,39.0,2.0,218.3,1,1
5,58,41.0,3.0,217.586,1,1
7,58,41.0,5.0,216.719,1,1
11,58,58.0,7.0,215.464,1,1
3,58,43.0,1.0,218.385,1,1
13,57,50.0,14.0,212.974,11,1
17,55,54.0,8.0,213.224,5,1
15,53,20.0,4.0,217.18,5,1
2,47,15.0,9.0,215.1,4,1
18,43,23.0,13.0,213.166,3,1


In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ["grid", "laps", "fastestLap", "rank", "fastestLapSpeed", "statusId"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

assembled_df = assembler.transform(final_df_fixed)

display(assembled_df.select("features", "target").limit(10))

features,target
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""58.0"",""39.0"",""2.0"",""218.3"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.0"",""58.0"",""41.0"",""3.0"",""217.586"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.0"",""58.0"",""41.0"",""5.0"",""216.719"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""11.0"",""58.0"",""58.0"",""7.0"",""215.464"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""58.0"",""43.0"",""1.0"",""218.385"",""1.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""13.0"",""57.0"",""50.0"",""14.0"",""212.974"",""11.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""17.0"",""55.0"",""54.0"",""8.0"",""213.224"",""5.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""15.0"",""53.0"",""20.0"",""4.0"",""217.18"",""5.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""47.0"",""15.0"",""9.0"",""215.1"",""4.0""]}",1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""18.0"",""43.0"",""23.0"",""13.0"",""213.166"",""3.0""]}",1


In [0]:
train_df, test_df = assembled_df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

Train rows: 6669
Test rows: 1583


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="target",
    featuresCol="features",
    numTrees=50,
    maxDepth=5,
    seed=42
)

rf_model = rf.fit(train_df)

predictions = rf_model.transform(test_df)

display(predictions.select("target", "prediction", "probability").limit(20))

target,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9077338092720262"",""0.09226619072797379""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9569872127496778"",""0.04301278725032224""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9086816765706044"",""0.0913183234293956""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.044245493611996575"",""0.9557545063880035""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.046016460361739515"",""0.9539835396382605""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.04771295085226173"",""0.9522870491477383""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7128187419505567"",""0.2871812580494433""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.04880637740145346"",""0.9511936225985466""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.04567451802715476"",""0.9543254819728452""]}"
1,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.04567451802715476"",""0.9543254819728452""]}"


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="weightedRecall"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Weighted Precision:", precision)
print("Weighted Recall:", recall)

Accuracy: 0.8610233733417562
F1 Score: 0.861088288680695
Weighted Precision: 0.8613292476852592
Weighted Recall: 0.8610233733417562


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="target",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print("AUC:", auc)

AUC: 0.9409415973303105


In [0]:
display(
    predictions.groupBy("target", "prediction").count().orderBy("target", "prediction")
)

target,prediction,count
0,0.0,718
0,1.0,118
1,0.0,102
1,1.0,645


In [0]:
import mlflow
import mlflow.spark

mlflow.set_experiment("/Users/jl7315@columbia.edu/homework4_f1_model_tracking")

experiment = mlflow.get_experiment_by_name("/Users/jl7315@columbia.edu/homework4_f1_model_tracking")
print(experiment)

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3489300533153131', creation_time=1776180185316, experiment_id='3489300533153131', last_update_time=1776180439781, lifecycle_stage='active', name='/Users/jl7315@columbia.edu/homework4_f1_model_tracking', tags={'mlflow.experiment.sourceName': '/Users/jl7315@columbia.edu/homework4_f1_model_tracking',
 'mlflow.experimentType': 'NOTEBOOK',
 'mlflow.ownerEmail': 'jl7315@columbia.edu',
 'mlflow.ownerId': '78292965468922'}>


In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/gr5069/raw/f1_data/mlflow_tmp"
print(os.environ["MLFLOW_DFS_TMP"])

/Volumes/gr5069/raw/f1_data/mlflow_tmp


In [0]:
import mlflow
import mlflow.spark
import pandas as pd
import matplotlib.pyplot as plt

with mlflow.start_run(run_name="rf_baseline_retry"):

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_param("seed", 42)
    mlflow.log_param("features", "grid,laps,fastestLap,rank,fastestLapSpeed,statusId")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("weighted_precision", precision)
    mlflow.log_metric("weighted_recall", recall)
    mlflow.log_metric("auc", auc)

    mlflow.spark.log_model(rf_model, "random_forest_model")

    confusion_pdf = predictions.groupBy("target", "prediction").count().toPandas()
    confusion_csv_path = "/tmp/confusion_matrix.csv"
    confusion_pdf.to_csv(confusion_csv_path, index=False)
    mlflow.log_artifact(confusion_csv_path)

    importance_pdf = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.featureImportances.toArray()
    })
    importance_csv_path = "/tmp/feature_importance.csv"
    importance_pdf.to_csv(importance_csv_path, index=False)
    mlflow.log_artifact(importance_csv_path)

    plt.figure(figsize=(8, 5))
    plt.bar(importance_pdf["feature"], importance_pdf["importance"])
    plt.xticks(rotation=45)
    plt.title("Feature Importance")
    plt.tight_layout()
    plot_path = "/tmp/feature_importance.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

print("Baseline run logged to MLflow.")

2026/04/14 17:59:53 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/14 17:59:57 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-8b134064-dc16-4d47-a7b4-79/tmpyprknneq/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/14 17:59:57 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

Baseline run logged to MLflow.


In [0]:
import mlflow
import mlflow.spark
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

param_grid = [
    {"numTrees": 20, "maxDepth": 3},
    {"numTrees": 20, "maxDepth": 5},
    {"numTrees": 20, "maxDepth": 8},
    {"numTrees": 50, "maxDepth": 3},
    {"numTrees": 50, "maxDepth": 5},
    {"numTrees": 50, "maxDepth": 8},
    {"numTrees": 100, "maxDepth": 3},
    {"numTrees": 100, "maxDepth": 5},
    {"numTrees": 100, "maxDepth": 8},
    {"numTrees": 150, "maxDepth": 5}
]

results_summary = []

for params in param_grid:
    rf = RandomForestClassifier(
        labelCol="target",
        featuresCol="features",
        numTrees=params["numTrees"],
        maxDepth=params["maxDepth"],
        seed=42
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    accuracy = MulticlassClassificationEvaluator(
        labelCol="target", predictionCol="prediction", metricName="accuracy"
    ).evaluate(preds)

    f1 = MulticlassClassificationEvaluator(
        labelCol="target", predictionCol="prediction", metricName="f1"
    ).evaluate(preds)

    precision = MulticlassClassificationEvaluator(
        labelCol="target", predictionCol="prediction", metricName="weightedPrecision"
    ).evaluate(preds)

    recall = MulticlassClassificationEvaluator(
        labelCol="target", predictionCol="prediction", metricName="weightedRecall"
    ).evaluate(preds)

    auc = BinaryClassificationEvaluator(
        labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(preds)

    run_name = f"rf_trees_{params['numTrees']}_depth_{params['maxDepth']}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_type", "RandomForestClassifier")
        mlflow.log_param("numTrees", params["numTrees"])
        mlflow.log_param("maxDepth", params["maxDepth"])
        mlflow.log_param("seed", 42)
        mlflow.log_param("features", "grid,laps,fastestLap,rank,fastestLapSpeed,statusId")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("weighted_precision", precision)
        mlflow.log_metric("weighted_recall", recall)
        mlflow.log_metric("auc", auc)

        mlflow.spark.log_model(model, f"random_forest_model_{params['numTrees']}_{params['maxDepth']}")

        confusion_pdf = preds.groupBy("target", "prediction").count().toPandas()
        confusion_csv_path = f"/tmp/confusion_matrix_{params['numTrees']}_{params['maxDepth']}.csv"
        confusion_pdf.to_csv(confusion_csv_path, index=False)
        mlflow.log_artifact(confusion_csv_path)

        importance_pdf = pd.DataFrame({
            "feature": feature_cols,
            "importance": model.featureImportances.toArray()
        })
        importance_csv_path = f"/tmp/feature_importance_{params['numTrees']}_{params['maxDepth']}.csv"
        importance_pdf.to_csv(importance_csv_path, index=False)
        mlflow.log_artifact(importance_csv_path)

        plt.figure(figsize=(8, 5))
        plt.bar(importance_pdf["feature"], importance_pdf["importance"])
        plt.xticks(rotation=45)
        plt.title(f"Feature Importance ({run_name})")
        plt.tight_layout()
        plot_path = f"/tmp/feature_importance_{params['numTrees']}_{params['maxDepth']}.png"
        plt.savefig(plot_path)
        plt.close()
        mlflow.log_artifact(plot_path)

    results_summary.append({
        "numTrees": params["numTrees"],
        "maxDepth": params["maxDepth"],
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "auc": auc
    })

results_pdf = pd.DataFrame(results_summary)
display(results_pdf.sort_values("auc", ascending=False))

2026/04/14 18:00:13 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/04/14 18:00:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-8b134064-dc16-4d47-a7b4-79/tmp4ijewvzx/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/04/14 18:00:16 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

numTrees,maxDepth,accuracy,f1,precision,recall,auc
100,8,0.8622867972204674,0.8623446306584239,0.8625374655010736,0.8622867972204675,0.9445565355520948
50,8,0.8622867972204674,0.8623376968346086,0.8624877964738988,0.8622867972204675,0.9440801483445743
20,8,0.8547062539481997,0.8547352080440516,0.8547887558367706,0.8547062539481995,0.9424548272836173
100,5,0.8578648136449779,0.8578491057412634,0.8578395751633376,0.8578648136449779,0.9420641097083703
150,5,0.8584965255843335,0.8584965255843335,0.8584965255843335,0.8584965255843335,0.941847133349986
50,5,0.8610233733417562,0.861088288680695,0.8613292476852592,0.8610233733417562,0.9409415973303102
20,5,0.8572331017056223,0.8572858691955116,0.8574372523674774,0.8572331017056223,0.9394659979631446
50,3,0.8515476942514214,0.8514688126883494,0.8515052206592372,0.8515476942514213,0.9349495269755255
100,3,0.85028427037271,0.8501131078601442,0.8503685132645555,0.85028427037271,0.9333954638330035
20,3,0.8515476942514214,0.8514952423344047,0.8514980161108882,0.8515476942514213,0.9307156857093446
